# Flight Tracker Pipeline — Data Exploration

This notebook pulls live aircraft position data from the OpenSky Network API and loads it into Postgres.

**Structure:**
1. Imports & configuration
2. Database connection (run once per session)
3. Create table (run once ever — safe to re-run, uses `IF NOT EXISTS`)
4. Fetch + load pipeline (run this repeatedly to ingest new snapshots)
5. Verification queries

## 1. Imports & configuration

In [2]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text

OPENSKY_URL = "https://opensky-network.org/api/states/all"

DB_HOST = "postgres"       # service name from docker-compose.yml, not localhost
DB_PORT = 5432
DB_NAME = "mydatabase"
DB_USER = "root"
DB_PASSWORD = "root"

# Raw column order returned by the OpenSky /states/all endpoint
OPENSKY_COLUMNS = [
    "icao24", "callsign", "origin_country", "time_position",
    "last_contact", "longitude", "latitude", "baro_altitude",
    "on_ground", "velocity", "true_track", "vertical_rate",
    "sensors", "geo_altitude", "squawk", "spi", "position_source"
]

## 2. Database connection

Run this once per session — every other cell in this notebook reuses `engine`.

In [3]:
engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Quick connectivity check
with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).fetchone()
print(version[0])

PostgreSQL 18.4 (Debian 18.4-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 3. Create table (one-time setup)

Safe to re-run any time — `IF NOT EXISTS` means it won't touch the table if it's already there.

In [4]:
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS flight_positions (
    id SERIAL PRIMARY KEY,
    icao24 VARCHAR(10),
    callsign VARCHAR(20),
    origin_country VARCHAR(100),
    time_position BIGINT,
    last_contact BIGINT,
    longitude DOUBLE PRECISION,
    latitude DOUBLE PRECISION,
    baro_altitude DOUBLE PRECISION,
    on_ground BOOLEAN,
    velocity DOUBLE PRECISION,
    true_track DOUBLE PRECISION,
    vertical_rate DOUBLE PRECISION,
    geo_altitude DOUBLE PRECISION,
    squawk VARCHAR(10),
    spi BOOLEAN,
    position_source INTEGER,
    ingested_at TIMESTAMP DEFAULT NOW()
);
"""

with engine.connect() as conn:
    conn.execute(text(CREATE_TABLE_SQL))
    conn.commit()

print("Table ready.")

Table ready.


## 4. Fetch + load pipeline

This is the part you re-run every time you want a fresh snapshot of live flights.

In [11]:
import inspect
print(inspect.getsource(load_states))

def load_states(df: pd.DataFrame) -> None:
    """Append a snapshot of aircraft states into Postgres, skipping exact duplicates."""
    df = df.where(pd.notnull(df), None)   # <-- convert all NaN -> None for SQL compatibility

    records = df.to_dict(orient="records")
    insert_sql = text("""
        INSERT INTO flight_positions
            (icao24, callsign, origin_country, time_position, last_contact,
             longitude, latitude, baro_altitude, on_ground, velocity,
             true_track, vertical_rate, geo_altitude, squawk, spi, position_source)
        VALUES
            (:icao24, :callsign, :origin_country, :time_position, :last_contact,
             :longitude, :latitude, :baro_altitude, :on_ground, :velocity,
             :true_track, :vertical_rate, :geo_altitude, :squawk, :spi, :position_source)
        ON CONFLICT (icao24, time_position) DO NOTHING
    """)

    with engine.connect() as conn:
        conn.execute(insert_sql, records)
        conn.commit()

    print(f

In [12]:
def fetch_states() -> pd.DataFrame:
    """Pull the current global aircraft state vector from OpenSky."""
    response = requests.get(OPENSKY_URL)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data["states"], columns=OPENSKY_COLUMNS)
    df["callsign"] = df["callsign"].str.strip()
    df = df.drop(columns=["sensors"])  # always None — not in the table schema

    print(f"Fetched {len(df)} aircraft at {pd.Timestamp.now()}")
    return df


import math

def load_states(df: pd.DataFrame) -> None:
    """Append a snapshot of aircraft states into Postgres, skipping exact duplicates."""
    records = df.to_dict(orient="records")

    # Replace NaN with None in each record — dicts can hold real None, unlike float64 columns
    for row in records:
        for key, value in row.items():
            if isinstance(value, float) and math.isnan(value):
                row[key] = None

    insert_sql = text("""
        INSERT INTO flight_positions
            (icao24, callsign, origin_country, time_position, last_contact,
             longitude, latitude, baro_altitude, on_ground, velocity,
             true_track, vertical_rate, geo_altitude, squawk, spi, position_source)
        VALUES
            (:icao24, :callsign, :origin_country, :time_position, :last_contact,
             :longitude, :latitude, :baro_altitude, :on_ground, :velocity,
             :true_track, :vertical_rate, :geo_altitude, :squawk, :spi, :position_source)
        ON CONFLICT (icao24, time_position) DO NOTHING
    """)

    with engine.connect() as conn:
        conn.execute(insert_sql, records)
        conn.commit()

    print(f"Attempted to load {len(df)} rows (duplicates skipped automatically).")

In [13]:
df = fetch_states()
load_states(df)
df.head()

Fetched 7583 aircraft at 2026-08-05 02:14:32.470449
Attempted to load 7583 rows (duplicates skipped automatically).


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,squawk,spi,position_source
0,80162c,AXB1929,India,1.785896e+09,1785896065,77.9072,13.1878,1584.96,False,87.98,270.67,-2.6,1554.48,None,False,0
1,801638,AXB2782,India,1.785896e+09,1785896064,91.0451,13.1944,9456.42,False,226.46,132.61,0.0,NaN,None,False,0
2,ac494e,CMD3,United States,1.785896e+09,1785896065,-122.3347,40.4730,830.58,False,61.36,159.89,0.0,792.48,None,False,0
3,39de4a,TVF26JT,France,NaN,1785896045,NaN,NaN,NaN,False,0.00,343.12,NaN,NaN,None,False,0
4,ab6fdd,AAL1425,United States,1.785896e+09,1785896064,-86.4979,38.3016,11277.60,False,236.47,45.71,0.0,11917.68,2212,False,0


## 5. Verification queries

In [ ]:
pd.read_sql("SELECT COUNT(*) FROM flight_positions;", engine)

In [ ]:
pd.read_sql(
    "SELECT * FROM flight_positions ORDER BY ingested_at DESC LIMIT 5;", engine
)